# Chapter 6: Build an index with verifiable provenance

**The question.** Chapter 6 stores each measured result as a *fact* that points back to the exact file and field it came from. This run's live answer tests (Chapter 7) gave these facts to an LLM as context and used them as the correct answers. Before trusting those tests, check that every fact really matches its source.

**Result.** All 18 facts match their source fields, and the source file is unchanged.

**Evidence.** Run 20260923T220909Z repeated the regional study end to end on two machines. `spark-fe12` reran the search with three new optimizer seeds (74101 to 74103), made one live LLM proposal and drew a fresh test sample of 200,000 new customers. `spark-ef84` ran 70 live answer tests and nine report drafts with `claude-sonnet-5`. The simulated customers and the frozen model are the same as in the companions, but the seeds, proposal and test sample are new, so do not mix these numbers with the companion's. Rerunning this notebook makes no LLM call.

## Setup and source checks

Finds the saved data for this run and checks, by its SHA-256 fingerprint, that the results file is unchanged.

In [1]:
from pathlib import Path
import json, hashlib, os, platform
import numpy as np
import pandas as pd

# Locate the data root: the directory whose relative layout the evidence records use.
MARKER = "book/notebooks/regional_case/companion.py"

def find_data_root():
    configured = os.environ.get("MODEL_VALIDATION_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        if not (root / MARKER).is_file():
            raise FileNotFoundError(f"MODEL_VALIDATION_ROOT must contain {MARKER}")
        return root, "MODEL_VALIDATION_ROOT"
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / MARKER).is_file():
            return path, "repository checkout"
    try:
        from aiv.companions import locate
    except ImportError:
        raise FileNotFoundError(
            "Companion data not found. Install the aiv package, open the notebook "
            "inside the repository checkout, or set MODEL_VALIDATION_ROOT.") from None
    return locate()

ROOT, DATA_SOURCE = find_data_root()
LIVE = ROOT / "book/experiments/regional_live"
R = ROOT / "book/experiments/regional_live/runs/20260923T220909Z"
print("Python:", platform.python_version())
print("Data root found from:", DATA_SOURCE)
print("Run:", R.relative_to(ROOT).as_posix())
read=lambda p: json.loads((R/p).read_text(encoding="utf-8"))
results=read('regional/results.json')
reference=read('reference.json')
seeds=reference['seeds']
assert hashlib.sha256((R/'regional/results.json').read_bytes()).hexdigest()==reference['source_sha256']


Python: 3.12.3
Data root found from: repository checkout
Run: book/experiments/regional_live/runs/20260923T220909Z


## Check every fact against its source

`reference.json` holds 18 facts: for each of the three seeds, the error (`brier`), contrast and share on new customers of the final group from the search with the new feature (`reflection`) and from the old search (`continuation`). For each fact the cell recomputes the source file's fingerprint, looks up the recorded field and checks that the value matches.

In [2]:
def resolve(record):
 source=R/record['source']
 assert hashlib.sha256(source.read_bytes()).hexdigest()==record['sha256']
 value=json.loads(source.read_text(encoding="utf-8"))
 for part in record['pointer'].strip('/').split('/'):
  value=value[part]
 assert value==record['value']
 return value
for record in reference['facts'].values(): resolve(record)
display(pd.DataFrame(reference['facts']).T)
print('All source pointers and hashes verified')

,value,source,sha256,pointer,unit
region:confirmation/reflection_74101/brier,0.085772,regional/results.json,9c80ebefdb314135d6f900ab0fee3505f3adb1863afd2c...,/confirmation/reflection_74101/brier,squared probability error
region:confirmation/reflection_74101/contrast,0.072514,regional/results.json,9c80ebefdb314135d6f900ab0fee3505f3adb1863afd2c...,/confirmation/reflection_74101/contrast,squared probability error
region:confirmation/reflection_74101/share,0.106785,regional/results.json,9c80ebefdb314135d6f900ab0fee3505f3adb1863afd2c...,/confirmation/reflection_74101/share,fraction
region:confirmation/reflection_74102/brier,0.071811,regional/results.json,9c80ebefdb314135d6f900ab0fee3505f3adb1863afd2c...,/confirmation/reflection_74102/brier,squared probability error
region:confirmation/reflection_74102/contrast,0.058639,regional/results.json,9c80ebefdb314135d6f900ab0fee3505f3adb1863afd2c...,/confirmation/reflection_74102/contrast,squared probability error
region:confirmation/reflection_74102/share,0.13352,regional/results.json,9c80ebefdb314135d6f900ab0fee3505f3adb1863afd2c...,/confirmation/reflection_74102/share,fraction
region:confirmation/reflection_74103/brier,0.084401,regional/results.json,9c80ebefdb314135d6f900ab0fee3505f3adb1863afd2c...,/confirmation/reflection_74103/brier,squared probability error
region:confirmation/reflection_74103/contrast,0.070908,regional/results.json,9c80ebefdb314135d6f900ab0fee3505f3adb1863afd2c...,/confirmation/reflection_74103/contrast,squared probability error
region:confirmation/reflection_74103/share,0.105895,regional/results.json,9c80ebefdb314135d6f900ab0fee3505f3adb1863afd2c...,/confirmation/reflection_74103/share,fraction
region:confirmation/continuation_74101/brier,0.074524,regional/results.json,9c80ebefdb314135d6f900ab0fee3505f3adb1863afd2c...,/confirmation/continuation_74101/brier,squared probability error


All source pointers and hashes verified


**Read the output.** **In one line: all 18 facts match the results file they point to.**

**Columns.** The row name identifies the fact, for example `region:confirmation/reflection_74101/contrast`. `value`: the recorded number. `source`: the file it came from. `sha256`: that file's fingerprint when the fact was recorded. `pointer`: the path to the field inside the file. `unit`: what the number measures.

**Notice.** The contrasts are 0.072514, 0.058639 and 0.070908 for the new-feature search and 0.05968, 0.059781 and 0.07181 for the old search; these are the answers the live tests expected. A matching pointer proves that the number is what the run recorded, not that the simulated customers behave like real ones.